# Validación temporal - XGBoost
## Autor: Mireya Ortega

Objetivo:
Entrenar un modelo XGBoost con datos de enero a octubre de 2026 y validar su capacidad predictiva sobre noviembre de 2026.

Esto simula el escenario real de la competencia, donde posteriormente se entrenará con enero-noviembre para predecir diciembre.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [2]:
train = pd.read_csv("../../../data/raw/train.csv")

print("Dimensiones:", train.shape)
display(train.head())

Dimensiones: (110100, 25)


,id_cliente,mes,edad,ingresos,ratio_deuda_ingresos,antiguedad_cuenta_meses,numero_productos,saldo_promedio,dias_ultima_transaccion,ocupacion,...,es_nuevo_cliente,tiene_prestamo,antiguedad_direccion_meses,visitas_web_ultimos_90_dias,distancia_sucursal_km,dispositivo_principal,dia_preferido_pago,tiene_seguro,dias_ultima_interaccion,objetivo
0,1,202601,26,42120.349663,0.367310,139,2,40949.765573,67,professional,...,False,False,222,10,9.898223,android,21,True,67,0
1,2,202601,56,89837.390440,0.338092,29,2,35233.622561,304,manager,...,False,False,58,2,9.311922,ios,20,True,304,0
2,3,202601,34,45951.715455,0.444587,21,2,39486.902998,220,clerical,...,False,False,8,6,12.188249,android,12,False,220,0
3,4,202601,62,54673.148516,0.243261,43,2,300.000000,110,manual,...,False,False,119,9,10.459796,ios,20,False,110,0
4,5,202601,37,100239.135609,0.230946,113,1,32066.808006,247,professional,...,False,False,25,6,36.502645,ios,13,False,247,1


In [4]:
print("Meses disponibles:")
print(sorted(train["mes"].unique()))

print("\nDistribución objetivo:")
print(train["objetivo"].value_counts())

print("\nProporción objetivo:")
print(train["objetivo"].value_counts(normalize=True))

Meses disponibles:
[np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608), np.int64(202609), np.int64(202610), np.int64(202611)]

Distribución objetivo:
objetivo
0    93533
1    16567
Name: count, dtype: int64

Proporción objetivo:
objetivo
0    0.849528
1    0.150472
Name: proportion, dtype: float64


In [5]:
train_fe = train.sort_values(["id_cliente", "mes"]).copy()

In [6]:
train_fe["meses_observado"] = (
    train_fe.groupby("id_cliente").cumcount() + 1
)

In [7]:
train_fe["grupo_antiguedad_cuenta"] = pd.cut(
    train_fe["antiguedad_cuenta_meses"],
    bins=[-1, 12, 36, 60, 120, 9999],
    labels=["0-12", "13-36", "37-60", "61-120", "120+"]
)

In [8]:
train_model = train_fe[train_fe["mes"] < 202611].copy()
validacion = train_fe[train_fe["mes"] == 202611].copy()

print("Entrenamiento:", train_model.shape)
print("Validación:", validacion.shape)

print("Meses entrenamiento:", sorted(train_model["mes"].unique()))
print("Mes validación:", validacion["mes"].unique())

Entrenamiento: (100600, 27)
Validación: (9500, 27)
Meses entrenamiento: [np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608), np.int64(202609), np.int64(202610)]
Mes validación: [202611]


In [9]:
X_train = train_model.drop(
    columns=["objetivo", "id_cliente", "mes"]
)

y_train = train_model["objetivo"]

X_val = validacion.drop(
    columns=["objetivo", "id_cliente", "mes"]
)

y_val = validacion["objetivo"]

In [10]:
categoricas = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(categoricas)

['ocupacion', 'region', 'canal_adquisicion', 'banda_riesgo', 'dispositivo_principal', 'grupo_antiguedad_cuenta']


In [11]:
X_train = pd.get_dummies(
    X_train,
    columns=categoricas,
    drop_first=False
)

X_val = pd.get_dummies(
    X_val,
    columns=categoricas,
    drop_first=False
)

X_train, X_val = X_train.align(
    X_val,
    join="left",
    axis=1,
    fill_value=0
)

In [12]:
modelo = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="auc"
)

modelo.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'auc'


In [13]:
pred_val = modelo.predict_proba(X_val)[:, 1]

In [14]:
auc = roc_auc_score(y_val, pred_val)
gini = 2 * auc - 1

print("ROC AUC:", auc)
print("Gini:", gini)

ROC AUC: 0.6118090267064571
Gini: 0.2236180534129142


In [15]:
resultado_noviembre = validacion[
    ["id_cliente", "objetivo"]
].copy()

resultado_noviembre["prediccion"] = pred_val

display(
    resultado_noviembre
    .sort_values("prediccion", ascending=False)
    .head(20)
)

,id_cliente,objetivo,prediccion
106577,20108,0,0.532330
109707,24236,0,0.472422
110013,24542,1,0.470046
104097,14744,0,0.461888
109867,24396,0,0.452466
105637,18433,0,0.448789
109048,23495,0,0.445655
108919,23334,0,0.445643
104121,14796,1,0.440331
108587,22941,0,0.439653


In [16]:
top20 = resultado_noviembre.sort_values(
    "prediccion",
    ascending=False
).head(20)

print("Conversiones reales dentro del Top 20:")
print(top20["objetivo"].sum())

Conversiones reales dentro del Top 20:
8


In [18]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

meses_validacion = [
    202606,
    202607,
    202608,
    202609,
    202610,
    202611
]

resultados_backtest = []

for mes_val in meses_validacion:

    print("=" * 50)
    print("Validando mes:", mes_val)

    train_bt = train_fe[train_fe["mes"] < mes_val].copy()
    val_bt = train_fe[train_fe["mes"] == mes_val].copy()

    X_train_bt = train_bt.drop(
        columns=["objetivo", "id_cliente", "mes"]
    )
    y_train_bt = train_bt["objetivo"]

    X_val_bt = val_bt.drop(
        columns=["objetivo", "id_cliente", "mes"]
    )
    y_val_bt = val_bt["objetivo"]

    categoricas = X_train_bt.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    X_train_bt = pd.get_dummies(
        X_train_bt,
        columns=categoricas,
        drop_first=False
    )

    X_val_bt = pd.get_dummies(
        X_val_bt,
        columns=categoricas,
        drop_first=False
    )

    X_train_bt, X_val_bt = X_train_bt.align(
        X_val_bt,
        join="left",
        axis=1,
        fill_value=0
    )

    modelo_bt = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )

    modelo_bt.fit(X_train_bt, y_train_bt)

    pred_bt = modelo_bt.predict_proba(X_val_bt)[:, 1]

    auc = roc_auc_score(y_val_bt, pred_bt)
    gini = 2 * auc - 1

    resultados_backtest.append({
        "mes": mes_val,
        "auc": auc,
        "gini": gini
    })

    print("AUC:", auc)
    print("Gini:", gini)

Validando mes: 202606
AUC: 0.6157798571428571
Gini: 0.2315597142857142
Validando mes: 202607
AUC: 0.6273282344788645
Gini: 0.2546564689577291
Validando mes: 202608
AUC: 0.6323861020862886
Gini: 0.2647722041725773
Validando mes: 202609
AUC: 0.6163606154840096
Gini: 0.2327212309680191
Validando mes: 202610
AUC: 0.6262140062584659
Gini: 0.2524280125169318
Validando mes: 202611
AUC: 0.6118090267064571
Gini: 0.2236180534129142


In [19]:
resultados_backtest = pd.DataFrame(resultados_backtest)

display(resultados_backtest)

print("AUC promedio:", resultados_backtest["auc"].mean())
print("Gini promedio:", resultados_backtest["gini"].mean())

print("Desviación AUC:", resultados_backtest["auc"].std())

,mes,auc,gini
0,202606,0.615780,0.231560
1,202607,0.627328,0.254656
2,202608,0.632386,0.264772
3,202609,0.616361,0.232721
4,202610,0.626214,0.252428
5,202611,0.611809,0.223618


AUC promedio: 0.6216463070261572
Gini promedio: 0.24329261405231428
Desviación AUC: 0.008094655362504925


## Reconstrucción controlada del baseline

In [20]:
import pandas as pd
import numpy as np

# Volvemos a cargar el train ORIGINAL
train_limpio = pd.read_csv("../../../data/raw/train.csv")

print(train_limpio.shape)
print(sorted(train_limpio["mes"].unique()))

(110100, 25)
[np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608), np.int64(202609), np.int64(202610), np.int64(202611)]


In [21]:
def crear_features(df, usar_historial=False):

    datos = df.sort_values(
        ["id_cliente", "mes"]
    ).copy()

    # Cuántos meses lleva apareciendo el cliente
    datos["meses_observado"] = (
        datos.groupby("id_cliente").cumcount() + 1
    )

    # Agrupación de antigüedad de la cuenta
    datos["grupo_antiguedad_cuenta"] = pd.cut(
        datos["antiguedad_cuenta_meses"],
        bins=[-1, 12, 36, 60, 120, 9999],
        labels=[
            "0-12",
            "13-36",
            "37-60",
            "61-120",
            "120+"
        ]
    )

    # Features históricas opcionales
    if usar_historial:

        datos["interaccion_anterior"] = (
            datos.groupby("id_cliente")[
                "dias_ultima_interaccion"
            ].shift(1)
        )

        datos["cambio_interaccion"] = (
            datos["dias_ultima_interaccion"]
            - datos["interaccion_anterior"]
        )

        datos["interaccion_anterior"] = (
            datos["interaccion_anterior"].fillna(0)
        )

        datos["cambio_interaccion"] = (
            datos["cambio_interaccion"].fillna(0)
        )

    return datos

In [22]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

def backtest_xgboost(datos_fe, parametros=None):

    meses_validacion = [
        202606,
        202607,
        202608,
        202609,
        202610,
        202611
    ]

    resultados = []

    if parametros is None:
        parametros = {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 5,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "eval_metric": "auc",
            "n_jobs": -1
        }

    for mes_val in meses_validacion:

        train_bt = datos_fe[
            datos_fe["mes"] < mes_val
        ].copy()

        val_bt = datos_fe[
            datos_fe["mes"] == mes_val
        ].copy()

        X_train = train_bt.drop(
            columns=["objetivo", "id_cliente", "mes"]
        )

        y_train = train_bt["objetivo"]

        X_val = val_bt.drop(
            columns=["objetivo", "id_cliente", "mes"]
        )

        y_val = val_bt["objetivo"]

        categoricas = X_train.select_dtypes(
            include=["object", "category"]
        ).columns.tolist()

        X_train = pd.get_dummies(
            X_train,
            columns=categoricas
        )

        X_val = pd.get_dummies(
            X_val,
            columns=categoricas
        )

        X_train, X_val = X_train.align(
            X_val,
            join="left",
            axis=1,
            fill_value=0
        )

        modelo = XGBClassifier(**parametros)

        modelo.fit(X_train, y_train)

        pred = modelo.predict_proba(X_val)[:, 1]

        auc = roc_auc_score(y_val, pred)
        gini = 2 * auc - 1

        resultados.append({
            "mes": mes_val,
            "auc": auc,
            "gini": gini
        })

    resultados = pd.DataFrame(resultados)

    return resultados

In [23]:
features_v1 = crear_features(
    train_limpio,
    usar_historial=False
)

resultado_v1 = backtest_xgboost(features_v1)

display(resultado_v1)

print(
    "AUC promedio:",
    resultado_v1["auc"].mean()
)

print(
    "Gini promedio:",
    resultado_v1["gini"].mean()
)

,mes,auc,gini
0,202606,0.615780,0.231560
1,202607,0.627328,0.254656
2,202608,0.632386,0.264772
3,202609,0.616361,0.232721
4,202610,0.626214,0.252428
5,202611,0.611809,0.223618


AUC promedio: 0.6216463070261572
Gini promedio: 0.24329261405231428


In [24]:
features_v2 = crear_features(
    train_limpio,
    usar_historial=True
)

resultado_v2 = backtest_xgboost(features_v2)

display(resultado_v2)

print(
    "AUC promedio:",
    resultado_v2["auc"].mean()
)

print(
    "Gini promedio:",
    resultado_v2["gini"].mean()
)

,mes,auc,gini
0,202606,0.612088,0.224177
1,202607,0.626882,0.253764
2,202608,0.624453,0.248905
3,202609,0.619049,0.238097
4,202610,0.629339,0.258679
5,202611,0.617012,0.234025


AUC promedio: 0.6214705781644527
Gini promedio: 0.24294115632890564


In [25]:
features_v2 = crear_features(
    train_limpio,
    usar_historial=True
)

resultado_v2 = backtest_xgboost(features_v2)

display(resultado_v2)

print(
    "AUC promedio:",
    resultado_v2["auc"].mean()
)

print(
    "Gini promedio:",
    resultado_v2["gini"].mean()
)

,mes,auc,gini
0,202606,0.612088,0.224177
1,202607,0.626882,0.253764
2,202608,0.624453,0.248905
3,202609,0.619049,0.238097
4,202610,0.629339,0.258679
5,202611,0.617012,0.234025


AUC promedio: 0.6214705781644527
Gini promedio: 0.24294115632890564


In [26]:
comparacion = pd.DataFrame({
    "modelo": [
        "V1 - meses + antiguedad",
        "V2 - meses + antiguedad + historial"
    ],

    "auc_promedio": [
        resultado_v1["auc"].mean(),
        resultado_v2["auc"].mean()
    ],

    "gini_promedio": [
        resultado_v1["gini"].mean(),
        resultado_v2["gini"].mean()
    ],

    "auc_noviembre": [
        resultado_v1.loc[
            resultado_v1["mes"] == 202611,
            "auc"
        ].iloc[0],

        resultado_v2.loc[
            resultado_v2["mes"] == 202611,
            "auc"
        ].iloc[0]
    ]
})

display(
    comparacion.sort_values(
        "auc_promedio",
        ascending=False
    )
)

,modelo,auc_promedio,gini_promedio,auc_noviembre
0,V1 - meses + antiguedad,0.621646,0.243293,0.611809
1,V2 - meses + antiguedad + historial,0.621471,0.242941,0.617012


## Tuning de hiperparámetros de XGBoost

In [27]:
configuraciones = [
    {
        "nombre": "Baseline",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 5,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Depth 3",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Depth 4",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 4,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Depth 6",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Learning 0.03",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 5,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Learning 0.02",
        "n_estimators": 700,
        "learning_rate": 0.02,
        "max_depth": 5,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Child 3",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 5,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "nombre": "Child 5",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 5,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }
]

In [28]:
resultados_tuning = []

for config in configuraciones:

    print("=" * 60)
    print("Probando:", config["nombre"])

    parametros = {
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_child_weight": config["min_child_weight"],
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "random_state": 42,
        "eval_metric": "auc",
        "n_jobs": -1
    }

    resultado = backtest_xgboost(
        features_v1,
        parametros
    )

    auc_promedio = resultado["auc"].mean()
    gini_promedio = resultado["gini"].mean()
    desviacion = resultado["auc"].std()

    auc_noviembre = resultado.loc[
        resultado["mes"] == 202611,
        "auc"
    ].iloc[0]

    resultados_tuning.append({
        "modelo": config["nombre"],
        "auc_promedio": auc_promedio,
        "gini_promedio": gini_promedio,
        "desviacion_auc": desviacion,
        "auc_noviembre": auc_noviembre
    })

    print("AUC promedio:", auc_promedio)

Probando: Baseline
AUC promedio: 0.6216463070261572
Probando: Depth 3
AUC promedio: 0.6244062909984072
Probando: Depth 4
AUC promedio: 0.6243598354413754
Probando: Depth 6
AUC promedio: 0.6181944024878336
Probando: Learning 0.03
AUC promedio: 0.62039192426055
Probando: Learning 0.02
AUC promedio: 0.6217148521517372
Probando: Child 3
AUC promedio: 0.6203332445712252
Probando: Child 5
AUC promedio: 0.6208891095916955


In [29]:
tabla_tuning = pd.DataFrame(resultados_tuning)

tabla_tuning = tabla_tuning.sort_values(
    "auc_promedio",
    ascending=False
).reset_index(drop=True)

display(tabla_tuning)

,modelo,auc_promedio,gini_promedio,desviacion_auc,auc_noviembre
0,Depth 3,0.624406,0.248813,0.005156,0.616534
1,Depth 4,0.624360,0.248720,0.008006,0.616809
2,Learning 0.02,0.621715,0.243430,0.007740,0.612111
3,Baseline,0.621646,0.243293,0.008095,0.611809
4,Child 5,0.620889,0.241778,0.008356,0.609359
5,Learning 0.03,0.620392,0.240784,0.008049,0.613506
6,Child 3,0.620333,0.240666,0.008372,0.609482
7,Depth 6,0.618194,0.236389,0.007935,0.611175


## Tuning fino alrededor de max_depth = 3

In [30]:
configuraciones_finas = [
    {
        "nombre": "Depth3_Base",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Depth3_400arboles",
        "n_estimators": 400,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Depth3_LR003",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Depth3_Child3",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Depth3_Sub09",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Depth3_Col09",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Depth3_RegL2",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 2
    },

    {
        "nombre": "Depth3_RegAlpha01",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1
    }
]

In [31]:
resultados_finos = []

for config in configuraciones_finas:

    print("=" * 60)
    print("Probando:", config["nombre"])

    parametros = {
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_child_weight": config["min_child_weight"],
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "reg_alpha": config["reg_alpha"],
        "reg_lambda": config["reg_lambda"],
        "random_state": 42,
        "eval_metric": "auc",
        "n_jobs": -1
    }

    resultado = backtest_xgboost(
        features_v1,
        parametros
    )

    resultados_finos.append({
        "modelo": config["nombre"],
        "auc_promedio": resultado["auc"].mean(),
        "gini_promedio": resultado["gini"].mean(),
        "desviacion_auc": resultado["auc"].std(),
        "auc_noviembre": resultado.loc[
            resultado["mes"] == 202611,
            "auc"
        ].iloc[0]
    })

Probando: Depth3_Base
Probando: Depth3_400arboles
Probando: Depth3_LR003
Probando: Depth3_Child3
Probando: Depth3_Sub09
Probando: Depth3_Col09
Probando: Depth3_RegL2
Probando: Depth3_RegAlpha01


In [32]:
tabla_fina = pd.DataFrame(resultados_finos)

tabla_fina = tabla_fina.sort_values(
    "auc_promedio",
    ascending=False
).reset_index(drop=True)

display(tabla_fina)

,modelo,auc_promedio,gini_promedio,desviacion_auc,auc_noviembre
0,Depth3_Col09,0.625824,0.251648,0.005525,0.617602
1,Depth3_LR003,0.625719,0.251439,0.006024,0.615872
2,Depth3_Sub09,0.625600,0.251201,0.006076,0.616922
3,Depth3_RegAlpha01,0.625043,0.250086,0.005844,0.615600
4,Depth3_RegL2,0.624795,0.249589,0.005686,0.617314
5,Depth3_Child3,0.624563,0.249126,0.005328,0.616223
6,Depth3_Base,0.624406,0.248813,0.005156,0.616534
7,Depth3_400arboles,0.623351,0.246702,0.005364,0.615366


## Combinación de los mejores hiperparámetros

In [33]:
configuraciones_combinadas = [
    {
        "nombre": "Col09",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Col09_Sub09",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Col09_LR003",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Col09_Sub09_LR003",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Col095",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.95,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "nombre": "Col100",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 1.0,
        "reg_alpha": 0,
        "reg_lambda": 1
    }
]

In [34]:
resultados_combinados = []

for config in configuraciones_combinadas:

    print("=" * 60)
    print("Probando:", config["nombre"])

    parametros = {
        **{k: v for k, v in config.items() if k != "nombre"},
        "random_state": 42,
        "eval_metric": "auc",
        "n_jobs": -1
    }

    resultado = backtest_xgboost(
        features_v1,
        parametros
    )

    resultados_combinados.append({
        "modelo": config["nombre"],
        "auc_promedio": resultado["auc"].mean(),
        "gini_promedio": resultado["gini"].mean(),
        "desviacion_auc": resultado["auc"].std(),
        "auc_noviembre": resultado.loc[
            resultado["mes"] == 202611,
            "auc"
        ].iloc[0]
    })

Probando: Col09
Probando: Col09_Sub09
Probando: Col09_LR003
Probando: Col09_Sub09_LR003
Probando: Col095
Probando: Col100


In [35]:
tabla_combinada = pd.DataFrame(resultados_combinados)

tabla_combinada = tabla_combinada.sort_values(
    "auc_promedio",
    ascending=False
).reset_index(drop=True)

display(tabla_combinada)

,modelo,auc_promedio,gini_promedio,desviacion_auc,auc_noviembre
0,Col09,0.625824,0.251648,0.005525,0.617602
1,Col100,0.625502,0.251003,0.005594,0.616940
2,Col09_LR003,0.625446,0.250892,0.006064,0.615928
3,Col09_Sub09_LR003,0.625374,0.250748,0.006714,0.615157
4,Col09_Sub09,0.625318,0.250637,0.006676,0.615874
5,Col095,0.625306,0.250611,0.005577,0.616416


## Feature Engineering V2 - Cambios temporales

In [36]:
def crear_features_temporales(df, columnas_historial):

    datos = df.sort_values(
        ["id_cliente", "mes"]
    ).copy()

    # Features que ya sabemos que funcionan
    datos["meses_observado"] = (
        datos.groupby("id_cliente").cumcount() + 1
    )

    datos["grupo_antiguedad_cuenta"] = pd.cut(
        datos["antiguedad_cuenta_meses"],
        bins=[-1, 12, 36, 60, 120, 9999],
        labels=[
            "0-12",
            "13-36",
            "37-60",
            "61-120",
            "120+"
        ]
    )

    # Creamos valor anterior y cambio
    for col in columnas_historial:

        anterior = datos.groupby(
            "id_cliente"
        )[col].shift(1)

        datos[f"{col}_anterior"] = anterior

        datos[f"{col}_cambio"] = (
            datos[col] - anterior
        )

        datos[f"{col}_anterior"] = (
            datos[f"{col}_anterior"].fillna(datos[col])
        )

        datos[f"{col}_cambio"] = (
            datos[f"{col}_cambio"].fillna(0)
        )

    return datos

In [37]:
pruebas_features = {
    
    "Baseline_Col09": [],

    "Saldo": [
        "saldo_promedio"
    ],

    "VisitasWeb": [
        "visitas_web_ultimos_90_dias"
    ],

    "UltimaTransaccion": [
        "dias_ultima_transaccion"
    ],

    "UltimaInteraccion": [
        "dias_ultima_interaccion"
    ],

    "Productos": [
        "numero_productos"
    ],

    "ComportamientoCompleto": [
        "saldo_promedio",
        "visitas_web_ultimos_90_dias",
        "dias_ultima_transaccion",
        "dias_ultima_interaccion",
        "numero_productos"
    ]
}

In [38]:
parametros_ganadores = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 3,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "reg_alpha": 0,
    "reg_lambda": 1,
    "random_state": 42,
    "eval_metric": "auc",
    "n_jobs": -1
}

In [39]:
resultados_features = []

for nombre, columnas in pruebas_features.items():

    print("=" * 60)
    print("Probando:", nombre)

    features_prueba = crear_features_temporales(
        train_limpio,
        columnas
    )

    resultado = backtest_xgboost(
        features_prueba,
        parametros_ganadores
    )

    resultados_features.append({
        "modelo": nombre,
        "auc_promedio": resultado["auc"].mean(),
        "gini_promedio": resultado["gini"].mean(),
        "desviacion_auc": resultado["auc"].std(),
        "auc_noviembre": resultado.loc[
            resultado["mes"] == 202611,
            "auc"
        ].iloc[0]
    })

Probando: Baseline_Col09
Probando: Saldo
Probando: VisitasWeb
Probando: UltimaTransaccion
Probando: UltimaInteraccion
Probando: Productos
Probando: ComportamientoCompleto


In [40]:
tabla_features = pd.DataFrame(
    resultados_features
).sort_values(
    "auc_promedio",
    ascending=False
).reset_index(drop=True)

display(tabla_features)

,modelo,auc_promedio,gini_promedio,desviacion_auc,auc_noviembre
0,ComportamientoCompleto,0.626309,0.252619,0.005885,0.617254
1,Productos,0.625921,0.251842,0.005849,0.616931
2,UltimaTransaccion,0.625903,0.251805,0.005471,0.616910
3,Baseline_Col09,0.625824,0.251648,0.005525,0.617602
4,UltimaInteraccion,0.625783,0.251566,0.005421,0.617073
5,VisitasWeb,0.625299,0.250598,0.006208,0.615418
6,Saldo,0.624799,0.249598,0.005860,0.615303


## Prueba de estabilidad con múltiples semillas

In [41]:
semillas = [13, 42, 73, 101, 2026]

columnas_comportamiento = [
    "saldo_promedio",
    "visitas_web_ultimos_90_dias",
    "dias_ultima_transaccion",
    "dias_ultima_interaccion",
    "numero_productos"
]

features_baseline = crear_features_temporales(
    train_limpio,
    []
)

features_comportamiento = crear_features_temporales(
    train_limpio,
    columnas_comportamiento
)

In [42]:
resultados_semillas = []

for seed in semillas:

    print("=" * 50)
    print("Semilla:", seed)

    parametros_seed = {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "random_state": seed,
        "eval_metric": "auc",
        "n_jobs": -1
    }

    # BASELINE
    r_base = backtest_xgboost(
        features_baseline,
        parametros_seed
    )

    resultados_semillas.append({
        "modelo": "Baseline",
        "semilla": seed,
        "auc_promedio": r_base["auc"].mean()
    })

    # COMPORTAMIENTO
    r_comp = backtest_xgboost(
        features_comportamiento,
        parametros_seed
    )

    resultados_semillas.append({
        "modelo": "ComportamientoCompleto",
        "semilla": seed,
        "auc_promedio": r_comp["auc"].mean()
    })

Semilla: 13
Semilla: 42
Semilla: 73
Semilla: 101
Semilla: 2026


In [43]:
tabla_semillas = pd.DataFrame(resultados_semillas)

display(tabla_semillas)

resumen_semillas = (
    tabla_semillas
    .groupby("modelo")["auc_promedio"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(resumen_semillas)

,modelo,semilla,auc_promedio
0,Baseline,13,0.624477
1,ComportamientoCompleto,13,0.625604
2,Baseline,42,0.625824
3,ComportamientoCompleto,42,0.626309
4,Baseline,73,0.625224
5,ComportamientoCompleto,73,0.625295
6,Baseline,101,0.625560
7,ComportamientoCompleto,101,0.625928
8,Baseline,2026,0.625302
9,ComportamientoCompleto,2026,0.625600


,mean,std,min,max
modelo,,,,
ComportamientoCompleto,0.625747,0.000386,0.625295,0.626309
Baseline,0.625277,0.000505,0.624477,0.625824


## Ensemble multisemilla

In [44]:
def backtest_multisemilla(
    datos_fe,
    semillas=[13, 42, 73, 101, 2026]
):

    meses_validacion = [
        202606,
        202607,
        202608,
        202609,
        202610,
        202611
    ]

    resultados = []

    for mes_val in meses_validacion:

        print("=" * 60)
        print("Mes:", mes_val)

        train_bt = datos_fe[
            datos_fe["mes"] < mes_val
        ].copy()

        val_bt = datos_fe[
            datos_fe["mes"] == mes_val
        ].copy()

        X_train = train_bt.drop(
            columns=["objetivo", "id_cliente", "mes"]
        )

        y_train = train_bt["objetivo"]

        X_val = val_bt.drop(
            columns=["objetivo", "id_cliente", "mes"]
        )

        y_val = val_bt["objetivo"]

        categoricas = X_train.select_dtypes(
            include=["object", "category"]
        ).columns.tolist()

        X_train = pd.get_dummies(
            X_train,
            columns=categoricas
        )

        X_val = pd.get_dummies(
            X_val,
            columns=categoricas
        )

        X_train, X_val = X_train.align(
            X_val,
            join="left",
            axis=1,
            fill_value=0
        )

        predicciones_semillas = []

        for seed in semillas:

            modelo = XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.9,
                reg_alpha=0,
                reg_lambda=1,
                random_state=seed,
                eval_metric="auc",
                n_jobs=-1
            )

            modelo.fit(
                X_train,
                y_train
            )

            pred = modelo.predict_proba(
                X_val
            )[:, 1]

            predicciones_semillas.append(pred)

        # Promedio de las 5 predicciones
        pred_ensemble = np.mean(
            predicciones_semillas,
            axis=0
        )

        auc = roc_auc_score(
            y_val,
            pred_ensemble
        )

        gini = 2 * auc - 1

        resultados.append({
            "mes": mes_val,
            "auc": auc,
            "gini": gini
        })

        print("AUC:", auc)
        print("Gini:", gini)

    return pd.DataFrame(resultados)

In [45]:
resultado_ensemble = backtest_multisemilla(
    features_comportamiento
)

display(resultado_ensemble)

Mes: 202606
AUC: 0.62157925
Gini: 0.24315850000000006
Mes: 202607
AUC: 0.6312916955402134
Gini: 0.2625833910804267
Mes: 202608
AUC: 0.6295976144780379
Gini: 0.2591952289560757
Mes: 202609
AUC: 0.6243824708860597
Gini: 0.24876494177211939
Mes: 202610
AUC: 0.6324206543939787
Gini: 0.2648413087879573
Mes: 202611
AUC: 0.6182816069168213
Gini: 0.23656321383364265


,mes,auc,gini
0,202606,0.621579,0.243159
1,202607,0.631292,0.262583
2,202608,0.629598,0.259195
3,202609,0.624382,0.248765
4,202610,0.632421,0.264841
5,202611,0.618282,0.236563


In [46]:
print(
    "AUC promedio ensemble:",
    resultado_ensemble["auc"].mean()
)

print(
    "Gini promedio ensemble:",
    resultado_ensemble["gini"].mean()
)

print(
    "Desviación AUC:",
    resultado_ensemble["auc"].std()
)

AUC promedio ensemble: 0.6262588820358518
Gini promedio ensemble: 0.2525177640717036
Desviación AUC: 0.00571840148646985


full

In [47]:
train_final = pd.read_csv("../../../data/raw/train.csv")
test_final = pd.read_csv("../../../data/raw/test.csv")

print(train_final.shape)
print(test_final.shape)

(110100, 25)
(9900, 24)


In [48]:
test_final["orden_original"] = np.arange(len(test_final))

train_final["es_test"] = 0
test_final["es_test"] = 1

In [49]:
datos_finales = pd.concat(
    [train_final, test_final],
    ignore_index=True,
    sort=False
)

In [50]:
columnas_comportamiento = [
    "saldo_promedio",
    "visitas_web_ultimos_90_dias",
    "dias_ultima_transaccion",
    "dias_ultima_interaccion",
    "numero_productos"
]

datos_finales_fe = crear_features_temporales(
    datos_finales,
    columnas_comportamiento
)

In [51]:
train_fe_final = datos_finales_fe[
    datos_finales_fe["es_test"] == 0
].copy()

test_fe_final = datos_finales_fe[
    datos_finales_fe["es_test"] == 1
].copy()

In [52]:
columnas_eliminar_train = [
    "objetivo",
    "id_cliente",
    "mes",
    "es_test",
    "orden_original"
]

columnas_eliminar_test = [
    "objetivo",
    "id_cliente",
    "mes",
    "es_test",
    "orden_original"
]

X_final = train_fe_final.drop(
    columns=columnas_eliminar_train,
    errors="ignore"
)

y_final = train_fe_final["objetivo"]

X_test_final = test_fe_final.drop(
    columns=columnas_eliminar_test,
    errors="ignore"
)

In [53]:
categoricas = X_final.select_dtypes(
    include=["object", "category"]
).columns.tolist()

X_final = pd.get_dummies(
    X_final,
    columns=categoricas
)

X_test_final = pd.get_dummies(
    X_test_final,
    columns=categoricas
)

X_final, X_test_final = X_final.align(
    X_test_final,
    join="left",
    axis=1,
    fill_value=0
)

print(X_final.shape)
print(X_test_final.shape)

(110100, 55)
(9900, 55)


In [54]:
semillas = [13, 42, 73, 101, 2026]

predicciones_finales = []

for seed in semillas:

    print("Entrenando seed:", seed)

    modelo_final = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_alpha=0,
        reg_lambda=1,
        random_state=seed,
        eval_metric="auc",
        n_jobs=-1
    )

    modelo_final.fit(
        X_final,
        y_final
    )

    pred_seed = modelo_final.predict_proba(
        X_test_final
    )[:, 1]

    predicciones_finales.append(pred_seed)

Entrenando seed: 13
Entrenando seed: 42
Entrenando seed: 73
Entrenando seed: 101
Entrenando seed: 2026


In [55]:
prediccion_ensemble = np.mean(
    predicciones_finales,
    axis=0
)

In [56]:
submission_v2 = pd.DataFrame({
    "id_cliente": test_fe_final["id_cliente"].values,
    "prediccion": prediccion_ensemble,
    "orden_original": test_fe_final["orden_original"].values
})

submission_v2 = (
    submission_v2
    .sort_values("orden_original")
    .drop(columns="orden_original")
    .reset_index(drop=True)
)

In [58]:
print("Filas:", submission_v2.shape)

print(
    "Orden correcto:",
    submission_v2["id_cliente"].equals(
        test_final["id_cliente"].reset_index(drop=True)
    )
)

print(
    "Nulos:",
    submission_v2["prediccion"].isna().sum()
)

print(
    "Predicción mínima:",
    submission_v2["prediccion"].min()
)

print(
    "Predicción máxima:",
    submission_v2["prediccion"].max()
)

Filas: (9900, 2)
Orden correcto: True
Nulos: 0
Predicción mínima: 0.017301727
Predicción máxima: 0.43130827


In [59]:
from pathlib import Path

ruta = Path("../../../resultados/predicciones")
ruta.mkdir(parents=True, exist_ok=True)

submission_v2.to_csv(
    ruta / "predicciones_Ortega_v2.csv",
    index=False
)

In [60]:
display(submission_v2.head(10))
display(
    submission_v2.sort_values(
        "prediccion",
        ascending=False
    ).head(20)
)

,id_cliente,prediccion
0,2,0.161402
1,3,0.076339
2,7,0.145883
3,10,0.065354
4,11,0.159234
5,12,0.029606
6,14,0.132616
7,16,0.142099
8,25,0.147272
9,32,0.120577


,id_cliente,prediccion
7445,23859,0.431308
6365,22277,0.425766
8104,24672,0.422492
6815,22941,0.422061
4761,19331,0.420765
5153,20108,0.417913
7631,24090,0.409599
7203,23495,0.404450
4708,19230,0.403685
6494,22455,0.402558


## Feature Engineering V3 - Estudio individual de nuevas señales

In [61]:
def crear_features_v3(df, extras=None):

    if extras is None:
        extras = []

    datos = df.sort_values(
        ["id_cliente", "mes"]
    ).copy()

    # Features que ya ganaron
    datos["meses_observado"] = (
        datos.groupby("id_cliente").cumcount() + 1
    )

    datos["grupo_antiguedad_cuenta"] = pd.cut(
        datos["antiguedad_cuenta_meses"],
        bins=[-1, 12, 36, 60, 120, 9999],
        labels=[
            "0-12",
            "13-36",
            "37-60",
            "61-120",
            "120+"
        ]
    )

    # Historial que ya sabemos que aporta algo
    columnas_comportamiento = [
        "saldo_promedio",
        "visitas_web_ultimos_90_dias",
        "dias_ultima_transaccion",
        "dias_ultima_interaccion",
        "numero_productos"
    ]

    for col in columnas_comportamiento:

        anterior = datos.groupby(
            "id_cliente"
        )[col].shift(1)

        datos[f"{col}_anterior"] = anterior.fillna(datos[col])

        datos[f"{col}_cambio"] = (
            datos[col] - anterior
        ).fillna(0)

    # 1. Observaciones previas
    if "observaciones_previas" in extras:
        datos["observaciones_previas"] = (
            datos.groupby("id_cliente").cumcount()
        )

    # 2. Meses desde primera observación
    if "meses_desde_primera" in extras:

        indice_mes = (
            (datos["mes"] // 100) * 12
            + datos["mes"] % 100
        )

        primero = indice_mes.groupby(
            datos["id_cliente"]
        ).transform("min")

        datos["meses_desde_primera"] = (
            indice_mes - primero
        )

    # 3. Saldo respecto a ingresos
    if "saldo_ingresos" in extras:
        datos["saldo_sobre_ingresos"] = (
            datos["saldo_promedio"]
            / datos["ingresos"].clip(lower=1)
        )

    # 4. Deuda estimada
    if "deuda_estimada" in extras:
        datos["deuda_estimada"] = (
            datos["ingresos"]
            * datos["ratio_deuda_ingresos"]
        )

    # 5. Cantidad de productos activos
    if "productos_activos" in extras:
        datos["productos_activos"] = (
            datos[
                [
                    "tiene_tarjeta_credito",
                    "tiene_prestamo",
                    "tiene_seguro"
                ]
            ]
            .astype(int)
            .sum(axis=1)
        )

    # 6. Promedio histórico de visitas
    if "historial_visitas" in extras:

        visitas_previas = (
            datos.groupby("id_cliente")[
                "visitas_web_ultimos_90_dias"
            ]
            .transform(
                lambda x: x.shift(1).expanding().mean()
            )
        )

        datos["promedio_visitas_previas"] = (
            visitas_previas.fillna(
                datos["visitas_web_ultimos_90_dias"]
            )
        )

        datos["visitas_vs_historial"] = (
            datos["visitas_web_ultimos_90_dias"]
            - datos["promedio_visitas_previas"]
        )

    # 7. Promedio histórico de saldo
    if "historial_saldo" in extras:

        saldo_previo = (
            datos.groupby("id_cliente")[
                "saldo_promedio"
            ]
            .transform(
                lambda x: x.shift(1).expanding().mean()
            )
        )

        datos["promedio_saldo_previo"] = (
            saldo_previo.fillna(
                datos["saldo_promedio"]
            )
        )

        datos["saldo_vs_historial"] = (
            datos["saldo_promedio"]
            - datos["promedio_saldo_previo"]
        )

    return datos

In [62]:
experimentos_v3 = {
    "Actual": [],

    "ObservacionesPrevias": [
        "observaciones_previas"
    ],

    "MesesDesdePrimera": [
        "meses_desde_primera"
    ],

    "SaldoIngresos": [
        "saldo_ingresos"
    ],

    "DeudaEstimada": [
        "deuda_estimada"
    ],

    "ProductosActivos": [
        "productos_activos"
    ],

    "HistorialVisitas": [
        "historial_visitas"
    ],

    "HistorialSaldo": [
        "historial_saldo"
    ]
}

In [63]:
resultados_v3 = []

for nombre, extras in experimentos_v3.items():

    print("=" * 60)
    print(nombre)

    datos_exp = crear_features_v3(
        train_limpio,
        extras
    )

    resultado = backtest_xgboost(
        datos_exp,
        parametros_ganadores
    )

    resultados_v3.append({
        "experimento": nombre,
        "auc_promedio": resultado["auc"].mean(),
        "gini_promedio": resultado["gini"].mean(),

        "auc_sep": resultado.loc[
            resultado["mes"] == 202609,
            "auc"
        ].iloc[0],

        "auc_oct": resultado.loc[
            resultado["mes"] == 202610,
            "auc"
        ].iloc[0],

        "auc_nov": resultado.loc[
            resultado["mes"] == 202611,
            "auc"
        ].iloc[0]
    })

Actual
ObservacionesPrevias
MesesDesdePrimera
SaldoIngresos
DeudaEstimada
ProductosActivos
HistorialVisitas
HistorialSaldo


In [64]:
tabla_v3 = pd.DataFrame(
    resultados_v3
).sort_values(
    "auc_promedio",
    ascending=False
).reset_index(drop=True)

display(tabla_v3)

,experimento,auc_promedio,gini_promedio,auc_sep,auc_oct,auc_nov
0,SaldoIngresos,0.626902,0.253803,0.627297,0.631709,0.617982
1,Actual,0.626309,0.252619,0.624681,0.633466,0.617254
2,HistorialVisitas,0.626185,0.252369,0.626096,0.631923,0.614887
3,ObservacionesPrevias,0.626101,0.252201,0.625529,0.631417,0.617972
4,MesesDesdePrimera,0.626101,0.252201,0.625529,0.631417,0.617972
5,HistorialSaldo,0.625931,0.251862,0.624590,0.632362,0.615811
6,DeudaEstimada,0.625891,0.251781,0.623350,0.630977,0.619596
7,ProductosActivos,0.625639,0.251278,0.624704,0.631728,0.616543


In [65]:
experimentos_combinados = {
    "SaldoIngresos": [
        "saldo_ingresos"
    ],

    "SaldoIngresos_Deuda": [
        "saldo_ingresos",
        "deuda_estimada"
    ],

    "SaldoIngresos_ObsPrevias": [
        "saldo_ingresos",
        "observaciones_previas"
    ],

    "SaldoIngresos_Deuda_ObsPrevias": [
        "saldo_ingresos",
        "deuda_estimada",
        "observaciones_previas"
    ]
}

In [66]:
resultados_combo = []

for nombre, extras in experimentos_combinados.items():

    print("=" * 60)
    print(nombre)

    datos_exp = crear_features_v3(
        train_limpio,
        extras
    )

    resultado = backtest_xgboost(
        datos_exp,
        parametros_ganadores
    )

    auc_sep = resultado.loc[
        resultado["mes"] == 202609,
        "auc"
    ].iloc[0]

    auc_oct = resultado.loc[
        resultado["mes"] == 202610,
        "auc"
    ].iloc[0]

    auc_nov = resultado.loc[
        resultado["mes"] == 202611,
        "auc"
    ].iloc[0]

    resultados_combo.append({
        "experimento": nombre,
        "auc_promedio": resultado["auc"].mean(),
        "gini_promedio": resultado["gini"].mean(),
        "auc_sep_oct": (auc_sep + auc_oct) / 2,
        "gini_sep_oct": 2 * ((auc_sep + auc_oct) / 2) - 1,
        "auc_nov": auc_nov
    })

tabla_combo = pd.DataFrame(
    resultados_combo
).sort_values(
    "auc_promedio",
    ascending=False
).reset_index(drop=True)

display(tabla_combo)

SaldoIngresos
SaldoIngresos_Deuda
SaldoIngresos_ObsPrevias
SaldoIngresos_Deuda_ObsPrevias


,experimento,auc_promedio,gini_promedio,auc_sep_oct,gini_sep_oct,auc_nov
0,SaldoIngresos,0.626902,0.253803,0.629503,0.259006,0.617982
1,SaldoIngresos_ObsPrevias,0.625930,0.251860,0.629218,0.258436,0.616125
2,SaldoIngresos_Deuda,0.625766,0.251531,0.627514,0.255027,0.616556
3,SaldoIngresos_Deuda_ObsPrevias,0.625584,0.251169,0.627900,0.255799,0.616947


In [67]:
features_saldo_ingresos = crear_features_v3(
    train_limpio,
    ["saldo_ingresos"]
)

resultado_ensemble_saldo = backtest_multisemilla(
    features_saldo_ingresos
)

display(resultado_ensemble_saldo)

Mes: 202606
AUC: 0.623559
Gini: 0.24711799999999995
Mes: 202607
AUC: 0.6303642772202688
Gini: 0.26072855444053755
Mes: 202608
AUC: 0.6299858884593723
Gini: 0.2599717769187446
Mes: 202609
AUC: 0.6261957658112053
Gini: 0.25239153162241057
Mes: 202610
AUC: 0.6324457930135085
Gini: 0.2648915860270169
Mes: 202611
AUC: 0.6188236862098838
Gini: 0.23764737241976763


,mes,auc,gini
0,202606,0.623559,0.247118
1,202607,0.630364,0.260729
2,202608,0.629986,0.259972
3,202609,0.626196,0.252392
4,202610,0.632446,0.264892
5,202611,0.618824,0.237647


In [68]:
auc_promedio_saldo = resultado_ensemble_saldo["auc"].mean()
gini_promedio_saldo = resultado_ensemble_saldo["gini"].mean()

auc_sep = resultado_ensemble_saldo.loc[
    resultado_ensemble_saldo["mes"] == 202609,
    "auc"
].iloc[0]

auc_oct = resultado_ensemble_saldo.loc[
    resultado_ensemble_saldo["mes"] == 202610,
    "auc"
].iloc[0]

auc_nov = resultado_ensemble_saldo.loc[
    resultado_ensemble_saldo["mes"] == 202611,
    "auc"
].iloc[0]

auc_sep_oct = (auc_sep + auc_oct) / 2
gini_sep_oct = 2 * auc_sep_oct - 1

print("AUC promedio Jun-Nov:", auc_promedio_saldo)
print("Gini promedio Jun-Nov:", gini_promedio_saldo)
print("AUC Sep-Oct:", auc_sep_oct)
print("Gini Sep-Oct:", gini_sep_oct)
print("AUC Noviembre:", auc_nov)
print("Gini Noviembre:", 2 * auc_nov - 1)

AUC promedio Jun-Nov: 0.6268957351190397
Gini promedio Jun-Nov: 0.2537914702380795
AUC Sep-Oct: 0.6293207794123569
Gini Sep-Oct: 0.25864155882471374
AUC Noviembre: 0.6188236862098838
Gini Noviembre: 0.23764737241976763


In [69]:
features_deuda = crear_features_v3(
    train_limpio,
    ["deuda_estimada"]
)

resultado_ensemble_deuda = backtest_multisemilla(
    features_deuda
)

display(resultado_ensemble_deuda)

Mes: 202606
AUC: 0.621432
Gini: 0.24286399999999997
Mes: 202607
AUC: 0.6302961784256859
Gini: 0.2605923568513717
Mes: 202608
AUC: 0.630237115125121
Gini: 0.2604742302502421
Mes: 202609
AUC: 0.6234627013385624
Gini: 0.24692540267712482
Mes: 202610
AUC: 0.6313177762391168
Gini: 0.26263555247823356
Mes: 202611
AUC: 0.6197994806625196
Gini: 0.2395989613250391


,mes,auc,gini
0,202606,0.621432,0.242864
1,202607,0.630296,0.260592
2,202608,0.630237,0.260474
3,202609,0.623463,0.246925
4,202610,0.631318,0.262636
5,202611,0.619799,0.239599


In [70]:
auc_promedio_deuda = resultado_ensemble_deuda["auc"].mean()
gini_promedio_deuda = resultado_ensemble_deuda["gini"].mean()

auc_sep_deuda = resultado_ensemble_deuda.loc[
    resultado_ensemble_deuda["mes"] == 202609,
    "auc"
].iloc[0]

auc_oct_deuda = resultado_ensemble_deuda.loc[
    resultado_ensemble_deuda["mes"] == 202610,
    "auc"
].iloc[0]

auc_nov_deuda = resultado_ensemble_deuda.loc[
    resultado_ensemble_deuda["mes"] == 202611,
    "auc"
].iloc[0]

auc_sep_oct_deuda = (
    auc_sep_deuda + auc_oct_deuda
) / 2

print("AUC promedio Jun-Nov:", auc_promedio_deuda)
print("Gini promedio Jun-Nov:", gini_promedio_deuda)
print("AUC Sep-Oct:", auc_sep_oct_deuda)
print("Gini Sep-Oct:", 2 * auc_sep_oct_deuda - 1)
print("AUC Noviembre:", auc_nov_deuda)
print("Gini Noviembre:", 2 * auc_nov_deuda - 1)

AUC promedio Jun-Nov: 0.6260908752985009
Gini promedio Jun-Nov: 0.2521817505970019
AUC Sep-Oct: 0.6273902387888396
Gini Sep-Oct: 0.2547804775776792
AUC Noviembre: 0.6197994806625196
Gini Noviembre: 0.2395989613250391


In [71]:
def backtest_multisemilla_predicciones(
    datos_fe,
    semillas=[13, 42, 73, 101, 2026]
):
    meses = [
        202606,
        202607,
        202608,
        202609,
        202610,
        202611
    ]

    resultados = {}

    for mes_val in meses:

        train_bt = datos_fe[
            datos_fe["mes"] < mes_val
        ].copy()

        val_bt = datos_fe[
            datos_fe["mes"] == mes_val
        ].copy()

        X_train = train_bt.drop(
            columns=["objetivo", "id_cliente", "mes"]
        )

        y_train = train_bt["objetivo"]

        X_val = val_bt.drop(
            columns=["objetivo", "id_cliente", "mes"]
        )

        y_val = val_bt["objetivo"].to_numpy()

        categoricas = X_train.select_dtypes(
            include=["object", "category"]
        ).columns.tolist()

        X_train = pd.get_dummies(
            X_train,
            columns=categoricas
        )

        X_val = pd.get_dummies(
            X_val,
            columns=categoricas
        )

        X_train, X_val = X_train.align(
            X_val,
            join="left",
            axis=1,
            fill_value=0
        )

        preds = []

        for seed in semillas:

            modelo = XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.9,
                reg_alpha=0,
                reg_lambda=1,
                random_state=seed,
                eval_metric="auc",
                n_jobs=-1
            )

            modelo.fit(X_train, y_train)

            preds.append(
                modelo.predict_proba(X_val)[:, 1]
            )

        resultados[mes_val] = {
            "y": y_val,
            "pred": np.mean(preds, axis=0)
        }

    return resultados

In [72]:
pred_saldo = backtest_multisemilla_predicciones(
    features_saldo_ingresos
)

pred_deuda = backtest_multisemilla_predicciones(
    features_deuda
)

In [73]:
pesos = np.arange(0, 1.01, 0.1)

resultados_blend = []

meses_desarrollo = [
    202606,
    202607,
    202608,
    202609,
    202610
]

for peso_saldo in pesos:

    aucs = []

    for mes in meses_desarrollo:

        y = pred_saldo[mes]["y"]

        pred_blend = (
            peso_saldo * pred_saldo[mes]["pred"]
            +
            (1 - peso_saldo) * pred_deuda[mes]["pred"]
        )

        aucs.append(
            roc_auc_score(y, pred_blend)
        )

    resultados_blend.append({
        "peso_saldo": peso_saldo,
        "peso_deuda": 1 - peso_saldo,
        "auc_promedio_jun_oct": np.mean(aucs)
    })

tabla_blend = pd.DataFrame(
    resultados_blend
).sort_values(
    "auc_promedio_jun_oct",
    ascending=False
)

display(tabla_blend)

,peso_saldo,peso_deuda,auc_promedio_jun_oct
10,1.0,0.0,0.628510
9,0.9,0.1,0.628441
8,0.8,0.2,0.628355
7,0.7,0.3,0.628263
6,0.6,0.4,0.628174
5,0.5,0.5,0.628069
4,0.4,0.6,0.627951
3,0.3,0.7,0.627817
2,0.2,0.8,0.627670
1,0.1,0.9,0.627518


In [74]:
mejor = tabla_blend.iloc[0]

peso_saldo = mejor["peso_saldo"]
peso_deuda = mejor["peso_deuda"]

print("Peso Saldo:", peso_saldo)
print("Peso Deuda:", peso_deuda)

Peso Saldo: 1.0
Peso Deuda: 0.0


In [75]:
mes = 202611

pred_nov_blend = (
    peso_saldo * pred_saldo[mes]["pred"]
    +
    peso_deuda * pred_deuda[mes]["pred"]
)

auc_nov_blend = roc_auc_score(
    pred_saldo[mes]["y"],
    pred_nov_blend
)

gini_nov_blend = 2 * auc_nov_blend - 1

print("AUC noviembre blend:", auc_nov_blend)
print("Gini noviembre blend:", gini_nov_blend)

AUC noviembre blend: 0.6188236862098838
Gini noviembre blend: 0.23764737241976763


In [76]:
print(mejor)
print("Peso saldo:", peso_saldo)
print("Peso deuda:", peso_deuda)

peso_saldo              1.00000
peso_deuda              0.00000
auc_promedio_jun_oct    0.62851
Name: 10, dtype: float64
Peso saldo: 1.0
Peso deuda: 0.0


In [77]:
%pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 407.0 kB/s eta 0:04:


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
from catboost import CatBoostClassifier

In [79]:
def backtest_catboost(
    datos_fe,
    parametros=None
):

    meses_validacion = [
        202606,
        202607,
        202608,
        202609,
        202610,
        202611
    ]

    resultados = []

    if parametros is None:
        parametros = {
            "iterations": 500,
            "learning_rate": 0.05,
            "depth": 6,
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "random_seed": 42,
            "verbose": 0
        }

    for mes_val in meses_validacion:

        print("=" * 50)
        print("Validando:", mes_val)

        train_bt = datos_fe[
            datos_fe["mes"] < mes_val
        ].copy()

        val_bt = datos_fe[
            datos_fe["mes"] == mes_val
        ].copy()

        X_train = train_bt.drop(
            columns=[
                "objetivo",
                "id_cliente",
                "mes"
            ]
        )

        y_train = train_bt["objetivo"]

        X_val = val_bt.drop(
            columns=[
                "objetivo",
                "id_cliente",
                "mes"
            ]
        )

        y_val = val_bt["objetivo"]

        categoricas = X_train.select_dtypes(
            include=["object", "category"]
        ).columns.tolist()

        # CatBoost necesita categóricas consistentes
        for col in categoricas:
            X_train[col] = X_train[col].astype(str)
            X_val[col] = X_val[col].astype(str)

        # Booleanos → enteros
        for col in X_train.select_dtypes(
            include=["bool"]
        ).columns:
            X_train[col] = X_train[col].astype(int)
            X_val[col] = X_val[col].astype(int)

        modelo = CatBoostClassifier(
            **parametros
        )

        modelo.fit(
            X_train,
            y_train,
            cat_features=categoricas
        )

        pred = modelo.predict_proba(
            X_val
        )[:, 1]

        auc = roc_auc_score(
            y_val,
            pred
        )

        gini = 2 * auc - 1

        resultados.append({
            "mes": mes_val,
            "auc": auc,
            "gini": gini
        })

        print("AUC:", auc)

    return pd.DataFrame(resultados)

In [80]:
features_catboost = crear_features_v3(
    train_limpio,
    ["saldo_ingresos"]
)

In [81]:
resultado_catboost = backtest_catboost(
    features_catboost
)

display(resultado_catboost)

Validando: 202606
AUC: 0.6248832857142856
Validando: 202607
AUC: 0.6279669601074523
Validando: 202608
AUC: 0.630404974015171
Validando: 202609
AUC: 0.6231595432029245
Validando: 202610
AUC: 0.6314194511013936
Validando: 202611
AUC: 0.6203209561147673


,mes,auc,gini
0,202606,0.624883,0.249767
1,202607,0.627967,0.255934
2,202608,0.630405,0.260810
3,202609,0.623160,0.246319
4,202610,0.631419,0.262839
5,202611,0.620321,0.240642


In [82]:
auc_cat = resultado_catboost["auc"].mean()
gini_cat = resultado_catboost["gini"].mean()

auc_sep_cat = resultado_catboost.loc[
    resultado_catboost["mes"] == 202609,
    "auc"
].iloc[0]

auc_oct_cat = resultado_catboost.loc[
    resultado_catboost["mes"] == 202610,
    "auc"
].iloc[0]

auc_nov_cat = resultado_catboost.loc[
    resultado_catboost["mes"] == 202611,
    "auc"
].iloc[0]

auc_sep_oct_cat = (
    auc_sep_cat + auc_oct_cat
) / 2

print("CATBOOST")
print("AUC promedio Jun-Nov:", auc_cat)
print("Gini promedio Jun-Nov:", gini_cat)

print("AUC Sep-Oct:", auc_sep_oct_cat)
print(
    "Gini Sep-Oct:",
    2 * auc_sep_oct_cat - 1
)

print("AUC Noviembre:", auc_nov_cat)
print(
    "Gini Noviembre:",
    2 * auc_nov_cat - 1
)

CATBOOST
AUC promedio Jun-Nov: 0.6263591950426658
Gini promedio Jun-Nov: 0.25271839008533153
AUC Sep-Oct: 0.6272894971521591
Gini Sep-Oct: 0.2545789943043182
AUC Noviembre: 0.6203209561147673
Gini Noviembre: 0.24064191222953468
